# Goal

Серия экспериментов. Треним за 5 поколений по 6 млн шагов с последующим отбором чемпионов. Наследники этих чемпионов будут использоваться на следующем шаге. 

Данный эксперимент - это первый шаг. 

Здесь агент должен научиться заходить в иглу (по типу `17e_study_2`). 

Учитываем неудачный опыт `17e_study_13_1`.

# set_hyperparameters

In [2]:
# @launchit.collect
def set_hyperparameters(HP, optuna_study, optuna_trial):
    generations_count = 6
    generation_ind = 0
    generation_steps_count = 6_000_000
    all_generations_steps_count = generation_steps_count * generations_count
    learn_rate_range = (0.00025, 0.00025 * 0.1)
    ent_coef_range = (0.05, 0.05 * 0.1)
    tau_range = (0.5, 0.1)
    ####
    
    import random
    HP.system.random_seed = random.randint(1, 100)
    HP.system.is_torch_deterministic = True
    HP.system.is_torch_compile = True
    
    HP.env.ident = 'FrostbiteNoFrameskip-v4'
    HP.env.count = 32 
    HP.env.is_episodic_life = True
    HP.env.actions_count = 6
    HP.env.idle_penalty = 0
    HP.env.life_lost_penalty = 0

    HP.agent.parent = None
    HP.agent.layers_count = 3 # number of transformer layers
    HP.agent.heads_count = 4 # number of heads used in multi-head attention
    HP.agent.d_model = 256 # dimension of the transformer
    HP.agent.obs_sequence_length = 4 # length observation chain agent incepts
    HP.agent.action_plan_length = 10 # number of actions agent must think upfront about
    HP.agent.positional_encoding = 'learned' # positional encoding type of the transformer: "", "absolute", "learned"
    
    # Video params
    HP.video.capture_policy = 'every(500000)' # video capture policy depending on steps
    HP.video.capture_env_rams = None
    HP.video.break_on_level_passed = False
    
    # Training procedure params (PPO related) 
    HP.ppo.global_steps_count = generation_steps_count # total number of steps 
    HP.ppo.rollout_steps_count = 512 # how many steps to run in a single policy rolllout
    HP.ppo.rollout_env_rams = [
        'com.develorium.neurolab.frostbite_ram:level1:1',
    ]
    HP.ppo.rollout_env_ram_patches = [
        ['last_life', 'full_igloo', 'bailey_right_at_the_igloo_door', 'temperature_10'],
        ['last_life', 'full_igloo', 'bailey_very_near_igloo_door', 'temperature_10'],  
        ['last_life', 'full_igloo', 'bailey_near_center', 'temperature_10'],
        ['last_life', 'one_remaining_igloo', 'bailey_near_center', 'temperature_10'],
        ['last_life', 'three_remaining_igloo', 'bailey_near_center', 'temperature_20'],
        ['last_life', 'half_igloo', 'bailey_near_center'],
        ['last_life', 'half_igloo'],
        ['last_life'],
    ]
    tau_change_speed = (tau_range[1] - tau_range[0]) / generations_count
    tau_a = tau_range[0] + tau_change_speed * generation_ind
    tau_b = tau_a + tau_change_speed
    HP.ppo.tau = f'linear({tau_a}, {tau_b})' # temperature to inject randomness during actions selection (Gumbel Max)
    
    HP.ppo.epochs_count = 2 
    HP.ppo.minibatches_count = 8
    learn_rate_change_speed = (learn_rate_range[1] - learn_rate_range[0]) / generations_count
    learn_rate_a = learn_rate_range[0] + learn_rate_change_speed * generation_ind
    learn_rate_b = learn_rate_a + learn_rate_change_speed
    HP.ppo.learn_rate = f'linear({learn_rate_a}, {learn_rate_b})'
    HP.ppo.optimizer = 'AdamW'
    
    HP.ppo.vf_coef = 0.5 # coefficient of the value function within loss function
    ent_coef_change_speed = (ent_coef_range[1] - ent_coef_range[0]) / generations_count
    ent_coef_a = ent_coef_range[0] + ent_coef_change_speed * generation_ind
    ent_coef_b = ent_coef_a + ent_coef_change_speed
    HP.ppo.ent_coef = f'linear({ent_coef_a}, {ent_coef_b})' # coefficient of the entropy member within loss function
    HP.ppo.consistency_coef = 0.1
    HP.ppo.prediction_coef = 0.1
    
    HP.ppo.gamma = 0.997 # return discount factor gamma
    HP.ppo.gae_lambda = 0.95 # lambda for the general advantage estimation
    HP.ppo.clip_coef = 0.1 # the surrogate clipping coefficient
    HP.ppo.clip_vloss = True # Toggles whether or not to use a clipped loss for the value function, as per the paper
    HP.ppo.max_grad_norm = 0.5 # the maximum norm for the gradient clipping
    HP.ppo.target_kl = None # e target KL divergence threshold
    HP.ppo.norm_adv = True # Toggles advantages normalization
    return HP
# @launchit.stop

In [5]:
from unittest.mock import Mock
HP = Mock()
set_hyperparameters(HP, None, None)
HP.ppo.tau, HP.ppo.learn_rate, HP.ppo.ent_coef

('linear(0.5, 0.43333333333333335)',
 'linear(0.00025, 0.00021250000000000002)',
 'linear(0.05, 0.0425)')

# Results
<TBD>

Провал. Ни одному запуску не удалось сделать прорыва (пройти хотя бы первый уровень). При этом было сделано 26 запусков. Для справки - в `17e_study_2` было 24 запуска, из которых 4 были прорывными.

Пока вижу такие причины, почему нет прорыва:
1) другая динамика аннилинга параметров `learn_rate`, `ent_coef`, `tau`
2) использование `HP.system.random_seed = random.randint(1, 100)` вместо `HP.system.random_seed = optuna_trial.suggest_int('HP.system.random_seed', 1, 100)`
3) баг в логике работе с RAM. Но тут вроде несколько раз проверял, да и по записанному видео видно, что RAM корректно применяется

Пока видится, что более вероятная - причина 1.

<img src="./img/reward.png">

**Вывод.** Перезапуск эксперимента с учётом причины 1.